# Valle nuevo

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.tsa as tsa
import statsmodels as sm
from datetime import datetime
import os
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

#Tensorflow
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential,save_model,load_model
from keras.layers import Dense
from keras.layers import LSTM
from keras.callbacks import EarlyStopping
from keras.metrics import MeanSquaredError
from keras.metrics import RootMeanSquaredError
from keras.optimizers import Adam
from sklearn.metrics import mean_squared_error
import keras

#Pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from keras.layers import Dropout, Input
from keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error

In [5]:
train = pd.read_csv("../../Datos/valle_nuevo_train.csv")
test = pd.read_csv("../../Datos/valle_nuevo_test.csv")

In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Fecha    147 non-null    object 
 1   Viajero  147 non-null    float64
dtypes: float64(1), object(1)
memory usage: 2.4+ KB


In [ ]:
tf.random.set_seed(123)
np.random.seed(123)

def crear_secuencias(datos_escalados, ventana):
    X, y = [], []
    for i in range(len(datos_escalados) - ventana):
        X.append(datos_escalados[i:i+ventana, 0])
        y.append(datos_escalados[i+ventana, 0])
    return np.array(X), np.array(y)

def construir_modelo(ventana, capas_unidades, dropout, lr):
    modelo = Sequential()
    modelo.add(Input(shape=(ventana, 1)))
    for idx, unidades in enumerate(capas_unidades):
        return_seq = idx < len(capas_unidades) - 1
        modelo.add(LSTM(unidades, return_sequences=return_seq))
        modelo.add(Dropout(dropout))
    modelo.add(Dense(1))
    modelo.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return modelo

def evaluar_config(train_serie, ventana, capas_unidades, dropout, lr, batch_size, epochs=100):
    """Entrena con un split interno 80/20 (cronológico) y devuelve RMSE/MAE de validación."""
    valores = train_serie["Viajero"].values.reshape(-1, 1).astype(float)

    corte = int(len(valores) * 0.8)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(valores[:corte])  # ajustar SOLO con la parte de entrenamiento real
    valores_esc = scaler.transform(valores)

    X, y = crear_secuencias(valores_esc, ventana)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    corte_secuencias = corte - ventana
    X_train, y_train = X[:corte_secuencias], y[:corte_secuencias]
    X_val, y_val = X[corte_secuencias:], y[corte_secuencias:]

    modelo = construir_modelo(ventana, capas_unidades, dropout, lr)
    es = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    modelo.fit(X_train, y_train, validation_data=(X_val, y_val),
               epochs=epochs, batch_size=batch_size, callbacks=[es], verbose=0)

    pred_val = scaler.inverse_transform(modelo.predict(X_val, verbose=0))
    real_val = scaler.inverse_transform(y_val.reshape(-1, 1))

    rmse = np.sqrt(mean_squared_error(real_val, pred_val))
    mae = mean_absolute_error(real_val, pred_val)
    return rmse, mae

# --- Grid de evidencia: variamos ventana, unidades, capas, dropout y lr ---
grid = [
    {'nombre': 'A: ventana6_1capa32',  'ventana': 6,  'capas_unidades': [32],     'dropout': 0.1, 'lr': 0.001,  'batch_size': 8},
    {'nombre': 'ventana12_1capa32',    'ventana': 12, 'capas_unidades': [32],     'dropout': 0.1, 'lr': 0.001,  'batch_size': 8},
    {'nombre': 'B: ventana12_1capa50', 'ventana': 12, 'capas_unidades': [50],     'dropout': 0.2, 'lr': 0.001,  'batch_size': 8},
    {'nombre': 'ventana6_2capas',      'ventana': 6,  'capas_unidades': [64, 32], 'dropout': 0.2, 'lr': 0.001,  'batch_size': 8},
    {'nombre': 'ventana12_2capas',     'ventana': 12, 'capas_unidades': [64, 32], 'dropout': 0.2, 'lr': 0.001,  'batch_size': 8},
    {'nombre': 'ventana12_2capas_lrbajo', 'ventana': 12, 'capas_unidades': [64, 32], 'dropout': 0.2, 'lr': 0.0005, 'batch_size': 16},
]

resultados = []
for cfg in grid:
    rmse, mae = evaluar_config(train_serie, cfg['ventana'], cfg['capas_unidades'],
                                cfg['dropout'], cfg['lr'], cfg['batch_size'])
    resultados.append({'Config': cfg['nombre'], 'RMSE_val': round(rmse, 2), 'MAE_val': round(mae, 2)})
    print(f"{cfg['nombre']:24s} | RMSE_val={rmse:9.2f} | MAE_val={mae:9.2f}")

df_resultados = pd.DataFrame(resultados).sort_values('RMSE_val')
print("\nRanking (menor RMSE de validación = mejor):")
print(df_resultados.to_string(index=False))

ValueError: could not convert string to float: '2009-01-01'

## LSTM 1

## LSTM 2